# Single Non-Interactive Distributed Simulation with pystclient

This example demonstrates how to run a **single non-interactive** distributed
simulation using an existing Spring-Mass-Damper (MSD) project, including
parameter configuration, execution, result collection, and plotting.


## 1 - Authenticate and select project


In [ ]:
from pystclient.clients import PyStclient

client = PyStclient()
_ = client.authenticate()

# Select an existing project by name
projects = client.project.info_all()
project_info = next(p for p in projects if p.name == "Tutorial")
print(f"Using project: {project_info.name}  (id={project_info.id})")

## 2 - Single Non-Interactive Distributed Simulation
A simulation is **non-interactive** when the simulator runs to completion
without user intervention (start -> run -> end). Here we create a single
distributed simulator with one parameter set, start it with recording, wait
for it to finish, and collect the results.


In [ ]:
from pystclient.models import (
    LoggingConfiguration,
    ModelParameters,
    ModelVariable,
    SimulationParameters,
)

# Configure simulation parameters with model-specific settings
sim_params = SimulationParameters(
    base_step_size=0.01,
    end_time=20,
    config_name="Config 1",
    model_parameters=[
        ModelParameters(
            name="Mass1",
            step_size=0.02,
            parameters=[ModelVariable(name="absTolerance", initial_value=2e-4)],
        ),
        ModelParameters(
            name="Spring1",
            parameters=[ModelVariable(name="absTolerance", initial_value=2e-4)],
        ),
        ModelParameters(
            name="Damper1",
            parameters=[ModelVariable(name="absTolerance", initial_value=2e-4)],
        ),
    ],
)
assert client.project.update_parameters(project_info.id, sim_params)

# Enable post-plotting so results are persisted
log_config = LoggingConfiguration(post_plotting=True)
assert client.project.update_log_config(project_info.id, log_config)

In [ ]:
import time

from pystclient.models import SimulationConfig
from pystclient.types import SimulationType

# Create a single non-interactive distributed simulator
statuses, future = client.simulator.create(
    SimulationConfig(
        project_id=project_info.id,
        parameter_set_names=["Config 1"],
        type=SimulationType.DISTRIBUTED,
    )
)
simulator_id = statuses[0].id
print(f"Simulator created: {simulator_id}  status={statuses[0].status}")

# Wait until the simulator is ready (node may need to scale up)
assert future.result(600), "Simulator did not become ready in time!"
print("Simulator is ready.")

# Poll until the simulation finishes
while not client.simulator.finished(simulator_id):
    s = client.simulator.status(simulator_id)
    print(f"  simulation_time={s.simulation_time}  end_time={s.end_time}")
    time.sleep(1)

print("Simulation finished.")

### 2.1 - Retrieve completed simulation


In [ ]:
from pystclient.models import SimulationInfo

# Wait for the simulation to appear in the completed list
completed_simulations: list[SimulationInfo] = []

while len(completed_simulations) != 1:
    completed_simulations = client.project.completed_simulations(
        project_id=project_info.id,
        simulator_ids=[simulator_id],
        limit=10,
    )
    time.sleep(5)

sim = completed_simulations[0]
print(f"  Simulation {sim.id}  name={sim.name}  param_set={sim.parameter_set_name}")

### 2.2 - Plot displacement from single simulation


In [ ]:
# pyright: reportUnknownMemberType=false
import matplotlib.pyplot as plt

from pystclient.models import MeasurementQuery, QueryVariable
from pystclient.types import FmuCausalityType
from pystclient.utils.time import convert_to_timestamp

sim = completed_simulations[0]
sim_measurements = client.measurement.measurements(project_info.id, sim.id)
assert sim_measurements, "No measurements found!"

measurement = sim_measurements[0]
q = MeasurementQuery(
    variables=[
        QueryVariable(
            instance_name="Spring1",
            name="dis_yx",
            causality=FmuCausalityType.INPUT,
        )
    ],
    time_from=0,
    time_to=10,
)
results = client.measurement.query(measurement.id, q)

fig, ax = plt.subplots(figsize=(10, 5))

for result in results:
    x = convert_to_timestamp(result.x)
    ax.plot(x, result.y, label=f"{sim.parameter_set_name}")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Displacement [m]")
ax.set_title("Single Simulation - Spring Displacement (dis_yx)")
ax.legend()
plt.tight_layout()
plt.show()